# 09. DQN과 현대 심층 강화학습

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/2038prob/defense-ai-textbook-code/blob/main/notebooks/%EC%8B%A4%EC%8A%B5_09_DQN%EA%B3%BC_%ED%98%84%EB%8C%80_%EC%8B%AC%EC%B8%B5_%EA%B0%95%ED%99%94%ED%95%99%EC%8A%B5.ipynb)

「국방 인공지능」 실습 코드 — 이 장의 파이썬 코드를 순서대로 모았습니다. **맨 위 setup 셀부터 위에서 아래로** 실행하세요(앞 셀에서 정의한 변수를 뒤 셀이 이어 씁니다). ⚠️ 표시가 붙은 셀은 개념 설명용 조각으로, 그대로는 실행되지 않을 수 있습니다.

전체 저장소: https://github.com/2038prob/defense-ai-textbook-code


In [ ]:
# === 실습 환경 준비 (Colab/로컬 공통) — 가장 먼저 실행하세요 ===
# 필요한 패키지를 설치하고(이미 있으면 건너뜀) 공통 유틸을 정의합니다.
import importlib.util, subprocess, sys
for pkg in ["torch", "torchvision", "numpy", "matplotlib", "scipy"]:
    if importlib.util.find_spec(pkg) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg])

import random, numpy as np, torch

def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch", torch.__version__, "| device:", device)

## 실습: CartPole에서 DQN 학습

### 🔬 실습 — 경험 리플레이 + 타깃 네트워크 DQN

① **Q-network와 리플레이 버퍼**: 상태를 받아 행동별 Q값 출력.


In [ ]:
import torch, torch.nn as nn, random
from collections import deque

class QNet(nn.Module):
    def __init__(self, state_dim, action_dim):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(state_dim, 128), nn.ReLU(),
                                 nn.Linear(128, 128), nn.ReLU(),
                                 nn.Linear(128, action_dim))
    def forward(self, x): return self.net(x)

buffer = deque(maxlen=10000)              # 리플레이 버퍼

② **온라인·타깃 네트워크 준비**: 타깃은 온라인 가중치로 초기화.


In [ ]:
import torch.optim as optim
online = QNet(4, 2); target = QNet(4, 2)
target.load_state_dict(online.state_dict())   # 동기화
optimizer = optim.Adam(online.parameters(), lr=1e-3)
gamma, C = 0.99, 100                          # 할인율, 타깃 갱신 주기

③ **학습 스텝**: 버퍼에서 미니배치를 뽑아 Huber 손실로 업데이트.


In [ ]:
def train_step(batch_size=64):
    if len(buffer) < batch_size: return
    s, a, r, s2, d = zip(*random.sample(buffer, batch_size))
    s  = torch.tensor(s, dtype=torch.float32);  s2 = torch.tensor(s2, dtype=torch.float32)
    a  = torch.tensor(a); r = torch.tensor(r, dtype=torch.float32)
    d  = torch.tensor(d, dtype=torch.float32)
    q  = online(s).gather(1, a.unsqueeze(1)).squeeze(1)
    with torch.no_grad():
        y = r + gamma * target(s2).max(1).values * (1 - d)   # 타깃 네트워크 사용
    loss = nn.functional.smooth_l1_loss(q, y)                # Huber 손실
    optimizer.zero_grad(); loss.backward()
    nn.utils.clip_grad_norm_(online.parameters(), 10)        # 기울기 클리핑
    optimizer.step()

④ **환경 루프**: ε-greedy로 행동→전이 저장→학습, C스텝마다 타깃 복사.


In [ ]:
import importlib.util, subprocess, sys
if importlib.util.find_spec("gymnasium") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gymnasium"])
import gymnasium as gym
env, eps, step = gym.make("CartPole-v1"), 1.0, 0
for ep in range(300):
    s, _ = env.reset(); done = False
    while not done:
        a = env.action_space.sample() if random.random() < eps \
            else int(online(torch.tensor(s, dtype=torch.float32)).argmax())
        s2, r, term, trunc, _ = env.step(a); done = term or trunc
        buffer.append((s, a, r, s2, float(term)))   # 진짜 종료(term)만 미래 가치 0
        s = s2; train_step(); step += 1
        if step % C == 0: target.load_state_dict(online.state_dict())
    eps = max(0.05, eps * 0.995)

## PPO-Clip: 좋은 변화도 과하면 추가 보상을 주지 않는다

PPO 튜닝에서 평균 반환 곡선 하나만 보면 원인 진단이 안 된다. 세 지표를 함께 기록하고 읽자. **(1) 근사 KL**: 한 업데이트에서 정책이 얼마나 움직였나. 대략 0.01$$0.05 수준을 유지하는 것이 흔한 관행이며, 훨씬 크면 학습률$$epoch 과다(정책 급변 위험), 거의 0인데 반환도 안 오르면 학습률 과소를 의심한다. **(2) clip fraction**: 미니배치에서 실제로 클리핑 영역에 들어간 샘플 비율. 대략 0.1$$0.3이 건강한 범위의 감각이고, 0.5를 넘으면 대부분의 샘플에서 이득이 포화되어 학습 신호가 약해졌다는 뜻이다(학습률$$epoch 축소). **(3) 엔트로피**: 정책의 탐색 여력. 급락하면 조기 수렴(한 행동에 몰림) 신호라 엔트로피 보너스를 키운다. 셋은 


> ⚠️ **설명용 조각** — 아래는 PPO 진단 지표(근사 KL·clip fraction) 계산을 보여주는 발췌입니다. `logp_new`, `logp_old`, `adv`, `target_kl` 등은 실제 PPO 학습 루프 안에서 정의되는 값이므로 이 셀만 단독 실행하면 NameError가 납니다.

In [ ]:
ratio = torch.exp(logp_new - logp_old)
approx_kl = 0.5 * ((logp_new - logp_old) ** 2).mean()   # KL 근사
clipped = torch.where(adv > 0, ratio > 1 + eps, ratio < 1 - eps)
clip_frac = clipped.float().mean()
# entropy는 정책 분포에서 직접: dist.entropy().mean()
if approx_kl > target_kl:   # 예: 0.05
    break                    # 이번 rollout의 추가 epoch 조기 종료

## GAE: 여러 스텝 TD 오차로 어드밴티지 만들기

### 🔬 실습 — 역순 재귀로 GAE 계산하기


In [ ]:
import torch

def compute_gae(rewards, values, dones, gamma=0.99, lam=0.95):
    """
    rewards: (T,)
    values:  (T+1,)  마지막 bootstrap value 포함
    dones:   (T,)    종료 transition이면 1
    """
    advantages = torch.zeros_like(rewards)
    gae = torch.tensor(0.0, device=rewards.device)
    for t in reversed(range(len(rewards))):
        nonterminal = 1.0 - dones[t]
        delta = rewards[t] + gamma * values[t+1] * nonterminal - values[t]
        gae = delta + gamma * lam * nonterminal * gae
        advantages[t] = gae
    returns = advantages + values[:-1]
    return advantages, returns

## PPO의 전체 손실과 업데이트 절차

### 🔬 실습 — PPO 미니배치 손실의 핵심 구현


In [ ]:
import torch
import torch.nn.functional as F

def ppo_loss(logp_new, logp_old, entropy, value_new,
             advantage, return_target, eps=0.2,
             value_coef=0.5, entropy_coef=0.01):
    ratio = torch.exp(logp_new - logp_old)
    surrogate1 = ratio * advantage
    surrogate2 = torch.clamp(ratio, 1-eps, 1+eps) * advantage
    policy_loss = -torch.min(surrogate1, surrogate2).mean()

    value_loss = 0.5 * F.mse_loss(value_new, return_target)
    entropy_bonus = entropy.mean()
    total = policy_loss + value_coef * value_loss - entropy_coef * entropy_bonus

    with torch.no_grad():
        approx_kl = (logp_old - logp_new).mean()
        clip_fraction = ((ratio - 1.0).abs() > eps).float().mean()
    metrics = {
        "policy_loss": policy_loss.item(),
        "value_loss": value_loss.item(),
        "entropy": entropy_bonus.item(),
        "approx_kl": approx_kl.item(),
        "clip_fraction": clip_fraction.item(),
    }
    return total, metrics

## Sim-to-Real과 강건성 평가

### 🔬 실습 — 여러 시드와 CVaR로 학습 결과 요약하기


In [ ]:
import numpy as np

# 각 행은 서로 다른 학습 시드, 열은 평가 에피소드 반환
returns = np.array([
    [90, 88, 91, 87, 20],
    [85, 86, 84, 82, 80],
    [93, 92, 90, 15, 89]
])
flat = returns.ravel()
mean = flat.mean()
std = flat.std(ddof=1)
q = np.quantile(flat, 0.05)
cvar_05 = flat[flat <= q].mean()                 # 최악 5% 평균
print(round(mean, 1), round(std, 1), round(cvar_05, 1))

---

실습 코드 저장소: https://github.com/2038prob/defense-ai-textbook-code  ·  교재 「국방 인공지능」
